# 04 · Protocolo de evaluación y E1
Lógica en `src/protocolo.py` (protocolo) y, más adelante, en el módulo de E1.

**Protocolo rolling-origin con reentrenamiento mensual.** Para cada mes *m* de 2024 (validación):
1. se entrena con las filas cuya etiqueta ya se conoce al empezar *m* (`target_month <= m - latencia`; con latencia 1, hasta el mes anterior);
2. se predicen todas las celdas de *m* a la vez;
3. se pasa al mes siguiente y se vuelve a entrenar con un mes más.

**2025 (test) no se toca.** Todo modelo (E0, E1…) pasa por este mismo protocolo para compararlo en igualdad.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

from baselines import indice_estacional, predecir_e0
from protocolo import MESES_VALIDACION, resumir, resumir_por_banda, resumir_por_mes

PROCESSED = ROOT / "data" / "processed"
MESES_VALIDACION

## Paneles
Los rezagos se calculan dentro de `predecir_e0` sobre el panel completo: cada fila del mes *m* solo usa meses anteriores (comprobado en los tests), así que no hay fuga aunque el panel incluya 2025.

In [ ]:
paneles = {n: pd.read_parquet(PROCESSED / f"panel_{n}.parquet") for n in ["1km", "2km", "distritos"]}
paneles["1km"].head()

## E0 por el protocolo
Los baselines no entrenan, pero se pasan por el mismo protocolo que usarán E1–E5. Los cinco de las directrices deben dar **lo mismo** que en el notebook 03.

El sexto, **`media_12_estacional`**, es la media de 12 meses multiplicada por el índice estacional del mes (ver más abajo). Es el **baseline a batir**.

In [ ]:
def evaluar_baselines(panel):
    preds = predecir_e0(panel, MESES_VALIDACION)
    tabla = pd.DataFrame({n: resumir(p) for n, p in preds.items()}).T
    return preds, tabla.round(3)

e0 = {n: evaluar_baselines(p) for n, p in paneles.items()}
e0["1km"][1]

In [ ]:
e0["2km"][1]

In [ ]:
e0["distritos"][1]

## Efecto agosto: índice estacional
Total de cada mes del año dividido entre la media mensual de su año, promediado sobre los años **completos ya conocidos** y **sin 2020** (COVID). Es lo que se sabe al empezar 2024: años 2019, 2021, 2022 y 2023. 2025 no interviene.

Agosto tiene alrededor de un tercio menos de accidentes todos los años (vacaciones); octubre–diciembre, algo más. La media de 12 meses predice casi lo mismo cada mes y por eso falla en agosto.

In [ ]:
indice = indice_estacional(paneles["1km"], pd.Timestamp("2023-12-01"))
indice.round(3).plot.bar(figsize=(7, 3), title="Índice estacional (1 = mes medio)")

## Errores por banda de frecuencia (1 km)
Banda según `ma_12` de la celda (media de los 12 meses anteriores): **0** si vale 0, **1–2** si está entre 0 y 3, **≥3** si es 3 o más. Nunca se usa el `y` del propio mes.

La banda 0 es trivial (todos predicen 0); en **≥3** es donde se distinguen los modelos.

In [ ]:
preds_1km = e0["1km"][0]
pd.concat(
    {n: resumir_por_banda(p).set_index("banda") for n, p in preds_1km.items()}
).round(3)

## Error mes a mes (1 km)
Media de 12 meses frente a su versión estacional: la corrección se nota sobre todo en agosto.

In [ ]:
por_mes = pd.DataFrame({
    n: resumir_por_mes(preds_1km[n]).set_index("target_month")["MAE"]
    for n in ["media_movil_12", "media_12_estacional"]
}).round(3)
por_mes

In [ ]:
por_mes.plot(marker="o", figsize=(7, 3), title="MAE por mes en 2024 (1 km)")

## E1 · GLM Poisson
*Pendiente:* se añadirá aquí con el mismo protocolo, **con el mes del año como variable** (para captar el efecto agosto), y se comparará con `media_12_estacional`, que es el baseline a batir.